# Ledger GL Workflow

This notebook walks thru the financial workflow to update the general ledger (journal) and product periodic (monthly, YE) reports. 

Refer to [GHub> IRSGuides.ChartofAccounts.md](https://github.com/wbgroupmgr/LLC-WB-Group/blob/main/pages/IRSGuide/LLC-ChartOfAccoounts.md)

In [1]:
# Load bookkeeping services
import os
from pathlib import Path
import datetime
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

from ledger.LLC import LLC
from ledger.llcCOA import ChartOfAccounts
from ledger.ledgerGeneral import ledgerGeneral
from ledger.llcBank import llcBank



top = Path.cwd().parents[2]
llc = LLC('WBGroupLLC',debug=False, top=top)  # debug='details'
# Load Latest/YE Bank Stmt
llc._Bank()


<h2> Chart of Accounts

In [2]:
# Show COA for reference
gl = ledgerGeneral(llc)
gl.coa

acctType     index                               acctID
Assets       Acct.Cash.Bank                      1010                       Cash in Bank (Operating Account)
             Acct.Cash.Security                  1020      1 Security Deposit Trust Account (Funds held f...
             Acct.Fixed.Depreciation.Accum       1460      Accumulated Depreciation (A "contra-asset" acc...
             Acct.Fixed.Land                     1410                                                   Land
             Acct.Fixed.Tangible                 1420      Buildings (The purchase price of the physical ...
             Acct.Fixed.Tangible.Improvements    1470      Buildings, Improvements (Major: new flooring, ...
             Acct.Fixed.Tangible.InConstruction  1310      Asset: Debit Buildings Not InService - purchas...
             Acct.Fixed.Tangible.InService       1430      Buildings InService (The purchase price of the...
Equities     Acct.Equity.Earnings.PnL            3100      Retained Earn

In [3]:
# Test COA access
cList = gl.coa.load()
gl.coa.save(cList)

print("COA Std Transaction Record keys:\n -- ",str(gl.coa.recCols()))

COA Std Transaction Record keys:
 --  ['dt', 'desc', 'amt', 'aType', 'acct', 'Ledger', 'acctMajor', 'acctMinor', 'acctSub', 'propNm', 'propID', 'propAddr', 'propOwners', 'tID', 'tDB', 'refDB', 'refDoc']


<h1> GL Workflow : Journal Entries into General Ledger

- e.g. Debit Cash $1,500; Credit Rental Income $1,500

<h2> WF 3.1 : llcAsset -> Journal (list of dict)

- llcAsset Account Types: Assets, Equity, Liability
- reconcile llcAsset, fix via assetEditor of llcAccess DB, major purchase/morgage/etc.)

In [4]:
# Normalize llcAsset fields - when changes only
aObj = llc.assets()

if False:
    ## ------ normalized llcAssets to coa.toRecDict fie4lds
    df = aObj.df.copy()
    df.rename(columns=dict(aID='tID', addr='propAddr', propRef='propID', stakeholderPct='propOwners'), inplace=True)
    
    df.drop(columns=['kwList'], inplace=True)
    df['tDB'] = 'llcAssets'
    refDict = {'Cash_LLC': 'Cash_LLC:llcBank Stmt',
     'H_805HighMesa': 'H_805HighMesa: Closing Docs 2025.08.26',
     'RV_RV1': 'RV_RV1: Bill of Sales'}
    dbDict = {'Cash_LLC': 'llcBank',
     'H_805HighMesa': 'llcAssets',
     'RV_RV1': 'llcAssets'}
    
    df['refDoc'] = df.apply(lambda r: refDict[r.propNm], axis=1)
    df['refDB'] = df.apply(lambda r: dbDict[r.propNm], axis=1)
    df.head(2)

    df = aObj.df
    aObj.save([gl.coa.toRecDict(**d) for d in df.to_dict(orient='records')])
aObj.df.head(2)                     

,dt,desc,amt,aType,acct,Ledger,acctMajor,acctMinor,acctSub,propNm,propID,propAddr,propOwners,tID,tDB,refDB,refDoc,_unknown
0,2025.08.20,YR.2025.Begining Balance: Cash,0.0,Debit,Acct.Cash.Bank,Acct.Equity.Earnings.PnL,NaN,NaN,NaN,Cash_LLC,a20250820-Cash1,"177 Kingsway Dr, Wimberley, 786767",{'o20250801_1': 100},a20250820-Cash1,llcAssets,llcBank,Cash_LLC:llcBank Stmt,{}
1,2025.08.20,Initial Investment by member,219000.0,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,NaN,NaN,NaN,H_805HighMesa,e20250826-805HMD,"177 Kingsway Dr, Wimberley, 786767",{'o20250801_1': 100},a20250826-Cash1,llcAssets,llcAssets,H_805HighMesa: Closing Docs 2025.08.26,{}


#### Create Temp and check accounts within llcAssets

In [5]:
# Create Temp lcAssets
from util.uiEditors import nbAssetEditor
ed = nbAssetEditor(llc)
tDF = pd.DataFrame(ed.loadTemp())
tDF.acct.unique()

array(['Acct.Cash.Bank', 'Acct.Equity.Owner.Cash',
       'Acct.Exp.Depreciation'], dtype=object)

# 3.1.1 - load llcAssets inter Journal

- llcAsset.toGL converts dual accounts into entry per account
- check if acct are within COA

In [6]:
# Class ledgerJournal
from ledger.ledgerGeneral import ledgerGeneral

class ledgerJournal(ledgerGeneral):
    def ToJournalDF(self, dbObj):
        df = dbObj.toGL(self)
        if df is None :
            print("Empty Journal or issue with DB")
            return None
        # --- Balance sheeet classification 
        df['acctType'] = df.acct.apply(lambda v : self.coa._Type(v))
        MajMin =  df.acct.apply(lambda v : gl.coa._Cat(v))
        df['acctMajor'] = [m[0] for m in MajMin]
        df['acctMinor'] = [m[1] for m in MajMin]
        return df

In [7]:
# llcAsset -> journalLedger -> Balance Sheet (primer)
aObj = llc.assets()
jl = ledgerJournal(llc)

a_jlDF = jl.ToJournalDF(aObj)
if a_jlDF is None: 
    bsDF = None
else:
    bsDF = jl.classifyAssets(a_jlDF)
bsDF

aType                                           Credit      Debit        Bal
acctType acct                                                               
Assets   Acct.Cash.Bank                      225909.84   224527.0   -1382.84
         Acct.Fixed.Depreciation.Accum         15000.0             -15000.00
         Acct.Fixed.Tangible.InConstruction                 177.0     177.00
         Acct.Fixed.Tangible.InService         1660.64   225674.5  224013.86
Equities Acct.Equity.Earnings.PnL                  0.0                  0.00
         Acct.Equity.Owner.Cash               224704.0    1660.64 -223043.36
Expense  Acct.Exp.Depreciation                            15000.0   15000.00
         Acct.Exp.Operating                                235.34     235.34
All      Total                               467274.48  467274.48       0.00

## 3.1.2 - Load llcExpRev to Journal; validate and remove dups 

In [8]:
# llcAsset -> journalLedger -> Balance Sheet (primer)
from ledger.llcExpRev import llcExpRev

erObj = llcExpRev(llc)
jl = ledgerJournal(llc)

er_jlDF = jl.ToJournalDF(erObj)
if er_jlDF is None: 
    bsDF = 'No Journal Data'
else:
    bsDF = jl.classifyAssets(a_jlDF)
bsDF

aType                                           Credit      Debit        Bal
acctType acct                                                               
Assets   Acct.Cash.Bank                      225909.84   224527.0   -1382.84
         Acct.Fixed.Depreciation.Accum         15000.0             -15000.00
         Acct.Fixed.Tangible.InConstruction                 177.0     177.00
         Acct.Fixed.Tangible.InService         1660.64   225674.5  224013.86
Equities Acct.Equity.Earnings.PnL                  0.0                  0.00
         Acct.Equity.Owner.Cash               224704.0    1660.64 -223043.36
Expense  Acct.Exp.Depreciation                            15000.0   15000.00
         Acct.Exp.Operating                                235.34     235.34
All      Total                               467274.48  467274.48       0.00

## 3.1.3 Load Bank Stmt into Journal; remove dups 

- Import Bk stmt and wrangle into Journal format [see COA.toRecDict)
- Identify new (transactions [Not in llcAssets or llcExpRev
- Store New ExpRev in llcExpRev DB
- *NOTE**: all llc DB's schema's are a superset of COE.toRecDict() fields


### 3.1.3.1 Map Raw Bk Stmt CSV -> COA RecDict (empty)

In [10]:
bk = llcBank(llc)

rawDF = bk._loadRawDF()
rawList = bk._loadRawList(rawDF)
print("rawDF, rawList:", len(rawDF), len(rawList))


rawDF, rawList: 54 54


### 3.1.3.2 Do heuristic pattern matching Bk.desc -> {acct, acctSub, TDesc}

In [11]:
descList = bk._loadDescList(rawDF.desc)
print("descList:", len(descList))

descList: 54


### 3.1.3.3 Merge COA BkRecDict x BkHeuristic transaction

In [12]:
workList = bk._loadWorkList(rawList, descList)
print("workList:", len(workList))

workList: 54


In [13]:
## Clean workList

if False:
    # Make a deepcopy 
    workList = [d.copy() for d in tList]
    
    # clean
    for tDict in workList:
        # Remove extra nodes from acct
        v = tDict['acct'].split('._')[0]
        tDict['acct']  = v
        # remove _unknown 
        del tDict['_unknown']
    workList[0]

### 3.1.3.5 Append `work` Transaction into llcExpRev DB

In [20]:
prevList = erObj.load()
print(f"ExpRev Prep: prev:{len(prevList)},   work:{len(workList)}")

erObj.save(prevList+workList)
newList = erObj.load()
print(f"ExpRev Updated: work:{len(workList)},   new:{len(newList)}")
set(d['acct'] for d in newList)

ExpRev Prep: prev:0,   work:54
ExpRev Updated: work:54,   new:54


{'Acct.Cash.Bank', 'Acct.Exp.Other', 'Acct.Exp.Repair', 'Acct.Exp.Util'}

In [19]:
# Recover to Prev, set to True to recover
if True:
    erObj.save([])

In [21]:
erDF = pd.DataFrame(newList)
print("erDF:", len(erDF))
erDF.head(2)

erDF: 54


,dt,desc,amt,aType,acct,Ledger,acctMajor,acctMinor,acctSub,propNm,propID,propAddr,propOwners,tID,tDB,refDB,refDoc,_unknown
0,2025.08.20,Owner investment,219000.0,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,NaN,NaN,NaN,H_805HighMesa,NaN,NaN,NaN,2025.08.20_219000.0,llcBank,llcBank,WT FED#02M03 NATIONAL FINANCIAL /ORG=FRANCIS X...,{}
1,2025.08.20,Owner Investment,50.0,Debit,Acct.Cash.Bank,Acct.Equity.Owner.Cash,NaN,NaN,NaN,H_805HighMesa,NaN,NaN,NaN,2025.08.20_50.0,llcBank,llcBank,WFB OPENING DEPOSIT FROM CARD XXXXXXXXXXXX1980...,{}


### 3.1.3.4 Load Bk ExpRev into Journal

In [17]:
x = erObj.load()
len(x)


0

In [23]:

# llcAsset -> journalLedger -> Balance Sheet (primer)
jl = ledgerJournal(llc)

a_jlDF = jl.ToJournalDF(erObj)
if a_jlDF is None: 
    bsBk_df = None
else:
    #  - bal sheet : Bank transactions
    bsBk_df = jl.classifyAssets(a_jlDF)
bsBk_df

aType                                      Credit      Debit           Bal
acctType acct                                                             
Assets   Acct.Cash.Bank                 214155.58   226522.5  1.236692e+04
         Acct.Fixed.Tangible.InService             214113.95  2.141140e+05
Equities Acct.Equity.Owner.Cash          219257.0            -2.192570e+05
Expense  Acct.Exp.Other                   1673.02       41.1 -1.631920e+03
         Acct.Exp.Repair                    135.0            -1.350000e+02
         Acct.Exp.Util                    1056.95            -1.056950e+03
Income   Acct.Rev.Other                    400.53       0.53 -4.000000e+02
         Acct.Rev.OwnerRent                4000.0            -4.000000e+03
All      Total                          440678.08  440678.08  1.182343e-11

In [ ]:
## Reconcile Prev and New llcExp



In [46]:
erDF = erObj.toDF().copy()
bk.coa._Type(erDF.acct)
#erDF #.acctType .acctType.isin(['Income', 'Expense'])
erDF.acct[0]

'Acct.Cash.Bank'

## OLD Stuff

In [ ]:
# llcBank - new approach

from ledger.ledgerDB import ledgerDB
from ledger.ledgerClassify import ledgerClassify

class llcBankStmt(ledgerDB):

    def foo(r, kList):
         
        if r.tID in kList:
            # 
            return '%%InAsset'
        else:
            return r.desc
        
    
    def filterAssets(self, bkDF):
        # Remove all transaction within Assets
        # Filter bk against Assets
        aDF =self.df.copy()
        kList = list(aDF.apply(lambda r : f"{aObj._key(r)[0]}_{r.aType}", axis=1))
        return kList


    def toGL(self, gl, **kwargs):

        df = self.toDF()
        
        # identify Mask cols of Bk
        df['amt'] = abs(df.amt)
        df['aType'] = df.TransType.apply(lambda v : 'Debit' if v == 'Rev' else 'Credit')
        df['tID'] = df.apply(lambda r : f"{aObj._key(r)[0]}_{r.aType}", axis=1)
    
        # isin_llcAssets
        kList = filterAssets(self.llc.assets(), df)
        df.desc = df.apply(lambda r: foo(r, []), axis=1)
    
        return df

In [42]:
# ledgerDB toGL test
def toGL(self, gl, **kwargs):
    '''
    Convert llc DB dual Accounts per transaction into journal format: 2 posts with 1 acct
    - DB must conform to COA.toRecDict fields
    - DB may be super set of fields 

    Needed: 
    - fromObj.toDF()(
    - fromObj._key(row)
    '''
    fromDF = self.toDF()
    if fromDF is None:
        return None

    # =========== Convert Dual accounts (acct, Ledger) into a single Acct columns        
    
    # ---- dfL make acct = Ledger; clear out Ledger
    dfL = fromDF.copy()
    dfL.acct = dfL.Ledger
    dfL.Ledger = np.nan
    #      Reverse sign of amt depending on aType [Debit/Credit]
    try:
        dfL.aType = fromDF.aType.apply(lambda v : 'Debit' if v == 'Credit' else 'Credit')
    except:
        pass
    
    # ---- df2 already has acct, clear out Ledger
    dfA = fromDF.copy()
    dfA.Ledger = np.nan
    
    df = pd.concat([dfA, dfL])

    # Return a GL with a single acct; 2 entries per transaction; sort by date via tID; reset index
    cols = [c for c in gl.coa.recCols() if c != 'Ledger']
    return df, cols
    return df[cols].sort_values(by='tID').reset_index(drop=True)

if False: glDF,cols = toGL(erObj, gl)
#erDF = erObj.toDF()
gl.coa.toRecDict()

coa = bk.coa
coa.toRecDict()

{'dt': nan,
 'desc': nan,
 'amt': nan,
 'aType': nan,
 'acct': nan,
 'Ledger': nan,
 'acctMajor': nan,
 'acctMinor': nan,
 'acctSub': nan,
 'propNm': nan,
 'propID': nan,
 'propAddr': nan,
 'propOwners': nan,
 'tID': nan,
 'tDB': nan,
 'refDB': nan,
 'refDoc': nan,
 '_unknown': {}}

In [48]:
pct = 0.05
tax = 40000
pymt1 = 20000
cost = (tax-pymt1)*pct
cost

1000.0

In [ ]:
erDF = erObj.load()

In [ ]:
# class llcExpRev - local version for construction
from ledger.llcExpRev import llcExpRev
class llcExpRev_Adapt(llcExpRev):
    def __init__(self, llc, **kwargs):
        super().__init__(llc, **kwargs)
        self.oID = 'llcExpRev'
        print("Debug InConst llcExp_Adapt - self.oID", self.oID)

    def load(self):
        return self.toDF().to_dict(orient='records')

    def newBk(self):
        '''
        Load Bk Stmt
        - remove transactions in llcAssets or llcExpRev
        '''
        df = self.llc.bk.df
        aDF = pd.DataFrame(self.llc.assets.load())

        

    def toDF(self):
        '''
        Import ExpRev
        '''
        return pd.DataFrame(self.load())

    def _wrangleExpDesc(self, r):
        '''
        Match common expenses based on KW's in Bk desc (refDoc)
        # Process repeating expenses
        # Handle special matches, venmo & 251022 ==> repaire
        '''
        if r.aType == 'Credit' : return r.desc
        d = r.refDoc
        expKWDict = {"comwsc": ['Acct.Exp.Util','Water','Pay Monthly Util'],
                     "pedernales" :['Acct.Exp.Util','Elec','Pay Monthly Util'],
                     "dispre.al" : ['Acct.Exp.Util','Waste','Pay Monthly Util'],
                     "allstate" : ['Acct.Exp.Util','Ins_Home','Pay Monthly Util'],
                     "check # 101" : ['Acct.Exp.Util','Water','Pay Monthly Util'],
                     "check # 102" : ['Acct.Exp.Util','Util','Pay Electrician Repair Outlet'],
                     "venmo&&251022" : ['Acct.Exp.Repair','Maintenance','Repair Utility Outlet,Electrician'],
                     "promotion bonus" : ['Acct.Rev.Other','Bank','Bank Promotion for account openning'],
                     "BankOpen WF opening deposit" : ['Acct.Equity.Owner.Cash','o20250801-1','Initial seed to open account'],
                    }
        for k,expDict in expKWDict.items():
            ## Special case of venmo payment
            if '&&' in k:
                kList = k.split('&&')
                k = kList[0]
                k2 = kList[1]
                if not( k2 in d ) : continue
            if k in d.lower() :
                if expDict[0] == r.acct : continue
                # Found match r.acct != match.acct
                # Append match KW to description -- fixed downstream 
                return f"{r.desc}%KW_{expDict}"
        return r.desc

    
    def _wrangleToDF(self, df):
        '''
        Wrangle Bk data - Common GL processing
        '''

        # ---- split acct into 2 parts: nodes & extra,  extra use ._Extra1._Extra2._ ....
        df['acctSub'] = df.acct.apply(lambda v : '.'.join(v.split('._')[1:]))
        # remove the extra nodes into acct
        df['acct'] = df.acct.apply(lambda v : v.split('._')[0])
        
        

    def bkToExpDB(self, gl, df, **kwargs):
        '''
        Map New bk transaction into expense transactions
        
        Import llc.bk.df 
        Wrangle into GL norm + Expense DB fields
        Merge New transactions into with llcExpense
        '''
        dbKey = gl.coa.dbNmDict['llcBank']
        
        
        # =========== wrangle / clean up single account DF

        df['amt'] = abs(df.amt)

        # Post as an increase to the Expense account
        df['aType'] = 'Debit'
        
        # ---- Wrangle final info per transaction                         
        df.rename(columns=dict(Acct='acct'), inplace=True)

        df['refDoc'] = df.desc
        df.desc = df.TDesc
        
        # --- Balance sheeet classification 
        df['acctType'] = df.acct.apply(lambda v : gl.coa._Type(v))
        #newDF['acctType'] = df.acct.apply(lambda v: coa._Type(v))

        ## ------ clean / wrangle columns to match GL normalized columns
        cList = list(set(gl.stdCols) & set(df.columns))
        newDF = df[cList].copy()
        

        ## ---- Add columns missing
        newDF['tID'] = df.apply(lambda r: f"{self._key(r)[0]}", axis=1)
        newDF['refDB'] = dbKey
        newDF['refKey'] = newDF.tID
        newDF['refProp'] = 'H_805HighMesa'

        ## --- handle acctSub info++
        self._wrangleToDF(newDF)

        '''
        Store Addt'n Expense reconcilation fields
        '''
        newDF['Ledger'] = 'Acct.Cash.Bank'   # default ledger
        newDF['Reconcile'] = 'No'
        newDF['Receipt'] = 'Bk Stmt'
        # This is unique to llcAsset

        # FIX - do this against new transactions        
        #df2['desc'] = df2.apply(lambda r : self._wrangleExpDesc(r), axis=1)
        
        #print('llcExpens.toGL complete:', len(newDF),len(cDF), len(df2))
        return newDF.sort_values(by='tID')[gl.stdCols]

    def toExpDB(self, glDF):
        
        return
        
    



In [ ]:
er = llcExpRev_Adapt(llc)
bkCols = ['tID', 'dt', 'amt', 'aType']
def newBk(self):
    '''
    Load Bk Stmt
    - remove transactions in llcAssets or llcExpRev
    '''
    aObj = self.llc.assets()
    
    df = self.llc.bk.df.copy()

    # identify Mask cols of Bk
    df['tID'] = df.apply(lambda r : aObj._key(r)[0], axis=1)
    df['amt'] = abs(df.amt)
    df['aType'] = df.TransType.apply(lambda v : 'Debit' if v == 'Rev' else 'Credit')
    
    # Filter bk against Assets
    aDF =llc.aObj.df.copy()
    aDF['tID'] = aDF.apply(lambda r : aObj._key(r)[0], axis=1)
    print('xxx', aDF.columns)
    #aDF[cols].sort_values(by='dt').head(10)

    # Columns to merge on for unique transaction
    mCols = ['tID', 'dt', 'amt', 'aType']
    onCol = ['tID', 'amt', 'aType']
    
    # Merge Bk with obj - aID will be nan if no match 
    mDF = pd.merge(df, aDF[mCols + ['aID']], on = onCol, how='left', suffixes=(None, '_r'))

    # now we have the tID that are new within the Bnk
    #mDF[mDF.aID.isna()]

    return mDF
    

bkDF = newBk(er)
print(len(bkDF))
    
bkDF.head(4)

In [ ]:
cols = ['tID', 'dt', 'amt', 'aType']
aDF =llc.aObj.df.copy()
aDF['tID'] = aDF.apply(lambda r : aObj._key(r)[0], axis=1)
aDF[cols].sort_values(by='dt').head(10)
print("aDF:", len(aDF))
aDF[bkCols].head(5)

In [ ]:
onCol = ['tID', 'amt', 'aType']
#b[cols].join(aDF[cols].set_index(onCol), on=onCol, how='inner', )
xdf = pd.merge(b, aDF, on = onCol, how='left')[['tID','aID']]
tIDNew = xdf.aID.isna()
print(len(b), len(aDF), len(tIDNew))
b['new'] = tIDNew
b.head(10)

In [ ]:
b

In [ ]:
# ??
llc.bk.df.columns#print(gl.stdCols)
eObj = llcExpenses_Adapt(llc)

eGL = ledgerGeneral(llc)

eDF = eObj.bkToExpDB(gl)
eObj.toExpDB(eDF)
print(f"Diff: {set(gl.stdCols) ^ set(eDF.columns)}")
llc.bk.df.columns

In [ ]:
eObj.save(eDF.to_dict(orient='records'))
#eDF[eDF.acct == 'Acct.Exp.Other'].iloc[1]

#eDF.apply(lambda r : f"{r.acct}_{r.x[0]}" if r.x != '' else r.acct , axis=1) #[eDF.desc.str.contains('%')]

In [ ]:
#sumAssets(aObj, toGL(aObj)) #, df=tDF))
assetSumDF = gl.classifyAcctType(eDF)

display(Markdown('<h2> GenLedger Summary '))
display(assetSumDF)
        

In [ ]:
#llc.bk.df[['dt', 'amt','Acct','AcctSub', 'TDesc']]
#

## 3.1 llcExpenses -> general ledger
- reconcile expenses with expenseEditor of llcExpenses DB via assetEditor
update llcCustomers DB - manually via json editor

In [ ]:
llc.bk.df.columns
    

## 3.2 classify bank transaction

## Run Report

In [ ]:
import pandas as pd
df = pd.DataFrame(llc.assets().load())

In [ ]:
sDF = df.groupby(['acct', 'aType']).amt.sum().unstack()
sDF.loc['Total'] = sDF.sum(axis=0)
sDF['diff'] = sDF.sum(axis=1)
sDF